In [20]:
import numpy as np 
import pandas as pd 
import torch
import torch.nn as nn 
import torch.optim as optim 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, log_loss
import xgboost as xgb 
import lightgbm as lgb 

In [21]:
np.random.seed(42)
n_samples = 5000

In [22]:
data = {
    'income': np.random.normal(50000, 15000, n_samples),
    'credit_score': np.random.normal(700, 50, n_samples),
    'debt_ratio': np.random.uniform(0.1, 0.9, n_samples),
    'state': np.random.choice(['NY', 'CA', 'TX', 'FL', 'IL'], n_samples),
    'employment_type': np.random.choice(['Full-time', 'Part-time', 'Self-employed', 'Unemployed'], n_samples)
}
df = pd.DataFrame(data)

In [23]:
df.head()

,income,credit_score,debt_ratio,state,employment_type
0,57450.712295,678.812016,0.357004,IL,Part-time
1,47926.035482,677.329295,0.182415,NY,Part-time
2,59715.328072,610.217841,0.157446,FL,Unemployed
3,72845.447846,683.495490,0.175421,NY,Full-time
4,46487.699379,736.641454,0.566295,CA,Full-time


In [24]:
logits = (
    -1.0
    + 0.00002 * (df['income'] - 50000)
    + 0.01 * (df['credit_score'] - 700)
    + 2.0 * (df['debt_ratio'] - 0.5)
)

probs = 1 / (1 + np.exp(-logits))

df['target'] = (
    np.random.rand(n_samples) < probs
).astype(int)

print(df['target'].value_counts())

target
0    3602
1    1398
Name: count, dtype: int64


In [25]:
cat_cols = ['state', 'employment_type']
num_cols = ['income', 'credit_score', 'debt_ratio']

In [26]:
cat_cardinalities = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    cat_cardinalities[col] = len(le.classes_)

In [27]:
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

X_cat = df[cat_cols].values
X_num = df[num_cols].values
y = df['target'].values

In [28]:
# 80% train + validation, 20% test
X_cat_train_val, X_cat_test, X_num_train_val, X_num_test, y_train_val, y_test = train_test_split(
    X_cat, X_num, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 80% of the remaining data for training, 20% for validation
X_cat_train, X_cat_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_cat_train_val, X_num_train_val, y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

print("Train:", np.unique(y_train, return_counts=True))
print("Validation:", np.unique(y_val, return_counts=True))
print("Test:", np.unique(y_test, return_counts=True))

Train: (array([0, 1]), array([2306,  894]))
Validation: (array([0, 1]), array([576, 224]))
Test: (array([0, 1]), array([720, 280]))


In [29]:
class TabularMLP(nn.Module):
    def __init__(self, cardinalities, embedding_dims, n_numericals, hidden_dims = [256, 64], dropout = 0.3):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, embed_dim) for num_classes, embed_dim in zip(cardinalities, embedding_dims) 
        ])


        total_emb_dim = sum(embedding_dims)
        input_dim = total_emb_dim + n_numericals

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim 

        layers.append(nn.Linear(prev_dim, 1)) # Binary output logit
        self.network = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embedded + [x_num], dim = 1)
        return self.network(x)


cardinalities_list = [cat_cardinalities[col] for col in cat_cols]
embedding_dims_list = [min(50, (c + 1) // 2) for c in cardinalities_list] # Rule of thumb for embedding size
model = TabularMLP(cardinalities_list, embedding_dims_list, len(num_cols))       


In [30]:
X_cat_t = torch.tensor(X_cat_train, dtype=torch.long)
X_num_t = torch.tensor(X_num_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

In [31]:
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

best_loss = float('inf')
patience, patience_counter = 5, 0

In [32]:
# Training with Early Stopping
for epoch in range(50):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_cat_t, X_num_t)
    loss = criterion(outputs, y_t)
    loss.backward()
    optimizer.step()
    
    # Validation check
    model.eval()
    with torch.no_grad():
        val_outputs = model(torch.tensor(X_cat_val, dtype=torch.long), torch.tensor(X_num_val, dtype=torch.float32))
        val_loss = criterion(val_outputs, torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)).item()
        
    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_model.pth'))

<All keys matched successfully>

In [33]:
# Train XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(pd.DataFrame(X_cat_train, columns=cat_cols).join(pd.DataFrame(X_num_train, columns=num_cols)), y_train)

# Train LightGBM
lgb_model = lgb.LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, verbose=-1)
lgb_model.fit(pd.DataFrame(X_cat_train, columns=cat_cols).join(pd.DataFrame(X_num_train, columns=num_cols)), y_train)

# Evaluate all models on Test Set
model.eval()
with torch.no_grad():
    mlp_probs = torch.sigmoid(model(torch.tensor(X_cat_test, dtype=torch.long), torch.tensor(X_num_test, dtype=torch.float32))).numpy()

xgb_probs = xgb_model.predict_proba(pd.DataFrame(X_cat_test, columns=cat_cols).join(pd.DataFrame(X_num_test, columns=num_cols)))[:, 1]
lgb_probs = lgb_model.predict_proba(pd.DataFrame(X_cat_test, columns=cat_cols).join(pd.DataFrame(X_num_test, columns=num_cols)))[:, 1]

print(f"MLP AUC-ROC:      {roc_auc_score(y_test, mlp_probs):.4f}")
print(f"XGBoost AUC-ROC:  {roc_auc_score(y_test, xgb_probs):.4f}")
print(f"LightGBM AUC-ROC: {roc_auc_score(y_test, lgb_probs):.4f}")

MLP AUC-ROC:      0.6978
XGBoost AUC-ROC:  0.7002
LightGBM AUC-ROC: 0.6972


#### Choose GBDTs (XGBoost / LightGBM / CatBoost) as your baseline for almost all standard tabular problems. They are faster to train, less sensitive to hyperparameter choices, and superior on small-to-medium datasets.

#### Choose MLPs with Entity Embeddings when your dataset features extremely high-cardinality categorical variables, requires online learning/streaming updates, or needs to be integrated into a larger multi-modal neural network architecture (e.g., combining user transaction history text with tabular user profiles).